# Smoke Test — Issue 13: `/analyze/custom` Endpoint

End-to-end test of the BYO RAG integration endpoint:
- `POST /analyze/custom` accepts caller-supplied question, answer, and chunks
- No ChromaDB or RAGBench involved — forensics run on the caller's data
- Response shape is identical to `POST /analyze`

**Prerequisites:**
- `backend/.env` contains `ANTHROPIC_API_KEY`
- Run from repo root: `cd backend && poetry run uvicorn main:app --port 8001`

**Acceptance criteria being verified:** all 7 listed at the bottom of this notebook.

In [ ]:
import requests
import json

BASE = "http://localhost:8001"

resp = requests.get(f"{BASE}/docs")
print("Server status:", resp.status_code, "(200 = up)")

## 1. Valid request returns 200 with `AnalyzeResponse` shape

In [ ]:
VALID_PAYLOAD = {
    "question": "What is the refund policy?",
    "answer": "Refunds are processed in 5-7 business days. Customers must request a refund within 30 days of purchase.",
    "chunks": [
        {"chunk_id": "doc_42_chunk_3", "text": "Refunds are processed within 5 to 7 business days of the return being received.", "score": 0.87},
        {"chunk_id": "doc_42_chunk_4", "text": "Customers must submit a refund request within 30 days of the original purchase date.", "score": 0.75},
        {"chunk_id": "doc_42_chunk_5", "text": "Refund eligibility requires the item to be in its original unopened packaging.", "score": 0.61},
    ]
}

r = requests.post(f"{BASE}/analyze/custom", json=VALID_PAYLOAD)
print("Status:", r.status_code, "(expected 200)")
assert r.status_code == 200, f"Expected 200, got {r.status_code}: {r.text[:300]}"
print("✓ 200 OK")

## 2. Response shape — all AnalyzeResponse keys present

In [ ]:
body = r.json()

EXPECTED_KEYS = [
    "question", "generated_answer", "retrieved_chunks",
    "ragas", "hedging_mismatch", "chunk_attribution",
    "retrieval_distribution", "embedding_space", "query_corpus_fit",
    "recommendation", "rule_id",
]
missing = [k for k in EXPECTED_KEYS if k not in body]
print("Keys present:", list(body.keys()))
print("Missing keys:", missing or "none — all present ✓")
assert not missing, f"Missing keys: {missing}"

## 3. Question and answer echoed back; chunks reflected

In [ ]:
assert body["question"] == VALID_PAYLOAD["question"], "question not echoed"
assert body["generated_answer"] == VALID_PAYLOAD["answer"], "answer not echoed"

expected_texts = [c["text"] for c in VALID_PAYLOAD["chunks"]]
assert body["retrieved_chunks"] == expected_texts, "retrieved_chunks don't match input chunk texts"

print("question        :", body["question"])
print("generated_answer:", body["generated_answer"])
print("retrieved_chunks:", body["retrieved_chunks"])
print("\n✓ question, answer, and chunks all echoed correctly")

## 4. All six forensics dimensions present and well-formed

In [ ]:
ragas = body["ragas"]
assert "retrieval_relevance_score" in ragas and "faithfulness_score" in ragas
assert "relevance_evidence" in ragas and "faithfulness_evidence" in ragas
print(f"ragas            : faith={ragas['faithfulness_score']:.3f}  relev={ragas['retrieval_relevance_score']:.3f}  ✓")

hm = body["hedging_mismatch"]
assert "overconfident_fraction" in hm and "underconfident_fraction" in hm and "claim_breakdown" in hm
print(f"hedging_mismatch : overconfident={hm['overconfident_fraction']:.2f}  underconfident={hm['underconfident_fraction']:.2f}  claims={hm['total_claims']}  ✓")

ca = body["chunk_attribution"]
assert "unattributed_fraction" in ca and "attribution_map" in ca and isinstance(ca["attribution_map"], list)
print(f"chunk_attribution: unattributed={ca['unattributed_fraction']:.2f}  mean_score={ca['mean_attribution_score']:.3f}  sentences={len(ca['attribution_map'])}  ✓")

rd = body["retrieval_distribution"]
assert "score_gap" in rd and "score_entropy" in rd and "n_chunks" in rd
print(f"retrieval_dist   : gap={rd['score_gap']:.3f}  entropy={rd['score_entropy']:.3f}  n_chunks={rd['n_chunks']}  ✓")

es = body["embedding_space"]
assert "centroid_distance" in es and "query_isolation" in es and "projection" in es
print(f"embedding_space  : centroid_dist={es['centroid_distance']:.3f}  query_isolation={es['query_isolation']:.3f}  points={len(es['projection'])}  ✓")

qcf = body["query_corpus_fit"]
assert "triggered" in qcf and "suggested_questions" in qcf
print(f"query_corpus_fit : triggered={qcf['triggered']}  mismatch_type={qcf['mismatch_type']}  ✓")

## 5. Recommendation and rule_id present

In [ ]:
VALID_RULE_IDS = {"R01", "R02", "R03", "R04", "R05", "R06", "R07", "R08", "R09"}

assert body["rule_id"] in VALID_RULE_IDS, f"Unknown rule_id: {body['rule_id']}"
assert len(body["recommendation"]) > 10, "recommendation suspiciously short"

print("rule_id       :", body["rule_id"])
print("recommendation:", body["recommendation"])
print("\n✓ rule fired and recommendation rendered")

## 6. Validation — empty chunks, missing fields

In [ ]:
# Empty chunks → 422
r_empty = requests.post(f"{BASE}/analyze/custom", json={**VALID_PAYLOAD, "chunks": []})
print("Empty chunks   :", r_empty.status_code, "(expected 422)", "✓" if r_empty.status_code == 422 else "✗")
assert r_empty.status_code == 422

# Missing question → 422
r_no_q = requests.post(f"{BASE}/analyze/custom", json={k: v for k, v in VALID_PAYLOAD.items() if k != "question"})
print("Missing question:", r_no_q.status_code, "(expected 422)", "✓" if r_no_q.status_code == 422 else "✗")
assert r_no_q.status_code == 422

# Missing answer → 422
r_no_a = requests.post(f"{BASE}/analyze/custom", json={k: v for k, v in VALID_PAYLOAD.items() if k != "answer"})
print("Missing answer  :", r_no_a.status_code, "(expected 422)", "✓" if r_no_a.status_code == 422 else "✗")
assert r_no_a.status_code == 422

# Chunk missing score → 400/422
r_no_score = requests.post(f"{BASE}/analyze/custom", json={
    **VALID_PAYLOAD,
    "chunks": [{"chunk_id": "c1", "text": "Some text."}]
})
print("Missing score   :", r_no_score.status_code, "(expected 400 or 422)", "✓" if r_no_score.status_code in (400, 422) else "✗")
assert r_no_score.status_code in (400, 422)

## 7. Unit test suite — all 12 pass

In [ ]:
import subprocess

proc = subprocess.run(
    ["poetry", "run", "pytest", "tests/test_analyze_custom.py", "-v", "--tb=short", "--no-header"],
    capture_output=True, text=True,
    cwd="../backend"
)
print(proc.stdout[-3000:])
if proc.returncode != 0:
    print(proc.stderr[-1000:])
assert proc.returncode == 0, "Unit tests failed!"
print("✓ All unit tests pass")

## Acceptance criteria

| # | Criterion | Verified by |
|---|-----------|-------------|
| 1 | `POST /analyze/custom` with valid inputs returns 200 with `AnalyzeResponse` | §1, §2 |
| 2 | Empty `chunks` list returns 422 | §6 |
| 3 | Missing `question` or `answer` returns 422 | §6 |
| 4 | Chunk missing `score` field returns 400/422 | §6 |
| 5 | Response shape identical to `/analyze` | §2 |
| 6 | Forensics services called with caller's chunks (question/answer/chunks echoed back) | §3, §4 |
| 7 | ChromaDB retriever NOT called (no `example_id` lookup) | Unit test 9 + no ChromaDB import in router |
| 8 | All six forensics dimensions present in response | §4 |
| 9 | `recommendation` and `rule_id` valid and present | §5 |